# 02 - Generation and pairwise files

Run this notebook after notebook 01 has completed. Nothing has to be edited.

1. **Import:** Kaggle -> Create -> New Notebook -> File -> Import Notebook -> upload this file.
2. **Settings** (right panel, Session options): Accelerator **GPU T4 x2** (one GPU is used), Internet **On**.
3. **Inputs:** **Add Input -> Notebooks ->** the saved, completed version of notebook 01 (several versions may be
   attached when the stages were split across sessions).
4. **Run:** **Save Version -> Save & Run All (Commit)**.
5. **Afterwards:** send the status of the version and its log; the last cell prints the first answers of every
   model, the generation table and the pair files. Then go on to notebook 03.

The notebook clones `https://github.com/zainahmadlums/genai-pa2.git` (branch `main`), reads the four checkpoints from the attached input and
writes `outputs/eval/generations/*.json` and `outputs/eval/pairs/*.jsonl`. Decoding is greedy with identical settings for every model.
All four finished runs are required: a missing checkpoint or a failing command raises and stops the run.
Finished generation files are reused on restart.

## 1. Get the code and install

In [ ]:
import os, shlex, shutil, subprocess, sys
from pathlib import Path

# ---- Settings: the defaults work as they are, nothing has to be edited ----------------------
REPO_URL = "https://github.com/zainahmadlums/genai-pa2.git"
REPO_BRANCH = "main"
CODE_DATASET_DIR = ""   # optional: use an uploaded copy of the code, e.g. "/kaggle/input/my-pa2-code/genaiPA2"
PREFER_ATTACHED_CODE = False   # True: take the code from an attached notebook output instead of GitHub
# ---------------------------------------------------------------------------------------------

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO_DIR = WORK / "small-model-alignment"
CODE_MARKER = "scripts/judge_pairwise_kaggle.py"   # present only in a complete copy of the code


def run(*argv, **popen_options):
    """Run one command given as separate arguments (no shell, so spaces in paths are safe), output shown live.

    A non-zero exit status raises CalledProcessError, which stops "Run All" at this cell.
    """
    argv = [str(arg) for arg in argv]
    print("$", shlex.join(argv), flush=True)
    with subprocess.Popen(argv, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace", **popen_options) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
    if process.returncode != 0:
        raise subprocess.CalledProcessError(process.returncode, argv)


def py(*argv, **popen_options):
    """Run a script or module with this notebook's Python interpreter."""
    run(sys.executable, *argv, **popen_options)


def find_code_in_inputs():
    """A copy of the code inside an attached input (an uploaded dataset or an earlier notebook's output)."""
    matches = sorted(INPUT.glob("**/" + CODE_MARKER))
    return matches[0].parents[1] if matches else None


if (REPO_DIR / CODE_MARKER).exists():
    print("using the code already at", REPO_DIR, "as it is (an existing copy is never updated)")
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a complete copy of the code. Delete it (or restart the "
                       "session) and run this cell again.")
else:
    if CODE_DATASET_DIR:
        source = Path(CODE_DATASET_DIR)
        if not (source / CODE_MARKER).exists():
            raise FileNotFoundError(f"CODE_DATASET_DIR={CODE_DATASET_DIR!r} does not contain {CODE_MARKER}")
    else:
        source = find_code_in_inputs() if PREFER_ATTACHED_CODE else None
    if source is not None:
        shutil.copytree(source, REPO_DIR, ignore=shutil.ignore_patterns("outputs", ".git", ".venv", "__pycache__"))
        print("copied the code from", source)
    else:
        if PREFER_ATTACHED_CODE:
            print("no copy of the code in the attached inputs; cloning it instead (needs Internet On)")
        run("git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR)

os.chdir(REPO_DIR)
os.environ["PYTHONUNBUFFERED"] = "1"   # training / generation logs appear line by line
sys.path.insert(0, str(REPO_DIR))
print("working directory:", os.getcwd())

In [ ]:
# Keeps Kaggle's CUDA build of PyTorch: torch is never installed or pinned here.
py("-m", "pip", "uninstall", "-y", "-q", "torchao")
py("-m", "pip", "install", "-q", "transformers>=4.56.2,<5", "datasets>=4.7,<5", "huggingface_hub",
   "accelerate>=1.4.0", "tqdm", "pyyaml", "sentencepiece", "safetensors", "matplotlib", "peft>=0.17.0",
   "trl==1.14.1", "pytest")
py("-m", "pip", "install", "-q", "-e", ".", "--no-deps")

In [ ]:
py("scripts/check_env.py", "--require_gpus", "1")

## 2. Locate the checkpoints from notebook 01 (all four are required)

In [ ]:
RUNS = {"sft": "best", "dpo_hh": "best", "dpo_ultrafeedback": "best", "persona_lora": "adapter"}   # run -> checkpoint


def finished(run_name):
    """Directory of a finished run: outputs/<run_name> in this session, else inside any attached input."""
    local = Path("outputs") / run_name
    if (local / "TRAINING_COMPLETE.json").exists():
        return local
    matches = sorted(INPUT.glob(f"**/outputs/{run_name}/TRAINING_COMPLETE.json"))
    return matches[-1].parent if matches else None


def checkpoint(run_name):
    """Checkpoint directory of a finished run. Raises when the run did not finish or its weights are missing."""
    run_dir = finished(run_name)
    if run_dir is None:
        raise FileNotFoundError(f"run '{run_name}' is not finished: no outputs/{run_name}/TRAINING_COMPLETE.json in "
                                "this session or in an attached input (Add Input -> Notebooks -> the saved version "
                                "of notebook 01 that completed it)")
    path = run_dir / RUNS[run_name]
    if not path.is_dir():
        raise FileNotFoundError(f"run '{run_name}' is marked finished but its checkpoint {path} is missing")
    return path


for name in RUNS:
    print(f"{name:18s} {finished(name) or 'not finished yet'}")

from alignment.io_utils import read_json

SFT = checkpoint("sft")
DPO_HH = checkpoint("dpo_hh")
DPO_UF = checkpoint("dpo_ultrafeedback")
PERSONA = checkpoint("persona_lora")
# The base model is prompted with the same template the SFT model was trained with.
TEMPLATE = read_json(SFT / "alignment_meta.json")["template"]
print("SFT:", SFT, "\nDPO_HH:", DPO_HH, "\nDPO_UF:", DPO_UF, "\nPERSONA:", PERSONA, "\ntemplate:", TEMPLATE)

## 3. Decoding settings (the same for every model in a comparison)

Greedy decoding, 256 new tokens. If the outputs loop, add an anti-repetition flag for **all** models as two more
entries of the `GEN_FLAGS` list below, for example `"--repetition_penalty", "1.1"` or
`"--no_repeat_ngram_size", "4"`, set `OVERWRITE = True` to regenerate, and record the change in the report.

A saved generation file is reused only for the same checkpoint contents, template and decoding flags. After any
such change the command stops and names the difference instead of reusing old answers; `OVERWRITE = True`
regenerates the files.

In [ ]:
OVERWRITE = False
RUN_SMOKE = False   # optional: True first runs every model on 5 prompts and prints the answers
GEN_FLAGS = ["--max_new_tokens", "256", "--batch_size", "16"] + (["--overwrite"] if OVERWRITE else [])   # one list entry per argument
GEN_DIR = "outputs/eval/generations"
ALPACA = "eval_sets/alpaca_eval_300_seed42.json"
HH = "eval_sets/hh_rlhf_eval_300_seed42.json"
PERSONA_PROMPTS = "eval_sets/persona_eval_prompts.json"
BASE = "Qwen/Qwen2.5-0.5B"
SMOL = "HuggingFaceTB/SmolLM2-360M-Instruct"


def generate(model, prompts, generator, out_name, *extra, smoke=False):
    py("scripts/generate.py", "--model_path", model, "--prompts_path", prompts, "--generator", generator,
       "--output_path", f"{GEN_DIR}/{out_name}", *extra, *GEN_FLAGS, *(["--smoke", "5"] if smoke else []))


def generate_all(smoke=False):
    generate(BASE, ALPACA, "qwen2.5-0.5b-base", "alpaca_base.json", "--template", TEMPLATE, smoke=smoke)
    generate(SFT, ALPACA, "qwen2.5-0.5b-sft", "alpaca_sft.json", smoke=smoke)
    generate(BASE, HH, "qwen2.5-0.5b-base", "hh_base.json", "--template", TEMPLATE, smoke=smoke)
    generate(SFT, HH, "qwen2.5-0.5b-sft", "hh_sft.json", smoke=smoke)
    generate(DPO_UF, ALPACA, "qwen2.5-0.5b-dpo-ultrafeedback", "alpaca_dpo_uf.json", smoke=smoke)
    generate(DPO_HH, HH, "qwen2.5-0.5b-dpo-hh", "hh_dpo_hh.json", smoke=smoke)
    generate(SMOL, PERSONA_PROMPTS, "smollm2-360m-instruct-base", "persona_base.json",
             "--template", "chat_template", smoke=smoke)
    generate(SMOL, PERSONA_PROMPTS, "smollm2-360m-instruct-persona-lora", "persona_lora.json",
             "--adapter_path", PERSONA, "--template", "chat_template", smoke=smoke)

## 4. Optional sanity check: 5 prompts per model

Off by default. With `RUN_SMOKE = True` it writes `*.smoke.json` files only and prints the answers.

In [ ]:
if RUN_SMOKE:
    generate_all(smoke=True)

## 5. Full generation

AlpacaEval subset (300 prompts, seed 42): base, SFT, UltraFeedback-DPO. Helpful/harmless set (300 prompts): base,
SFT, HH-DPO. Persona prompts (40): original SmolLM2-Instruct and the LoRA persona model.

In [ ]:
generate_all()

## 6. Pairwise files (written before any judging)

Response order is randomised with seed 42 (balanced A/B); model identities are stored only in each row's `hidden`
block. The persona pair is built twice: once to judge style faithfulness and once to check that usefulness and
safety held.

Pairing stops with an explanation if one of the two generation files is unfinished or if they were generated with
different template, decoding flags or precision: regenerate the affected file (`OVERWRITE = True`) and rerun.

In [ ]:
PAIRS_DIR = "outputs/eval/pairs"


def pair(baseline, candidate, comparison, profile):
    py("scripts/build_pairs.py", "--baseline_path", f"{GEN_DIR}/{baseline}",
       "--candidate_path", f"{GEN_DIR}/{candidate}", "--output_path", f"{PAIRS_DIR}/{comparison}_pairs.jsonl",
       "--comparison", comparison, "--profile", profile)


pair("alpaca_sft.json", "alpaca_dpo_uf.json", "dpo_uf_vs_sft", "general")
pair("hh_sft.json", "hh_dpo_hh.json", "dpo_hh_vs_sft", "safety")
pair("persona_base.json", "persona_lora.json", "persona_vs_base_style", "persona")
pair("persona_base.json", "persona_lora.json", "persona_vs_base_quality", "safety")
# Optional, for your own analysis (not a required comparison): SFT against the untuned base model.
pair("alpaca_base.json", "alpaca_sft.json", "sft_vs_base", "general")

## 7. Package and summary

In [ ]:
py("scripts/summarize_results.py", "--runs", "--eval_dir", "outputs/eval", "--out_dir", "outputs/report")
py("scripts/package_outputs.py", "--source", "outputs/eval", "outputs/report",
   "--zip_path", WORK / "eval_generations_and_pairs.zip")
for path in sorted(Path(GEN_DIR).glob("*")) + sorted(Path(PAIRS_DIR).glob("*")):
    print(f"{path.stat().st_size:>13,d}  {path}")
# The first two answers of every model, to see at a glance that they are sensible and that they end.
for path in sorted(Path(GEN_DIR).glob("*.json")):
    if not path.name.endswith((".meta.json", ".smoke.json")):
        print("=" * 100)
        for entry in read_json(path)[:2]:
            print(f"{path.name} [{entry['id']}] {entry['instruction'][:200]!r}\n  -> {entry['output'][:500]!r}")
print(Path("outputs/report/report_tables.md").read_text(encoding="utf-8"))
print("NOTEBOOK 02 FINISHED. Pair files:", sorted(path.name for path in Path(PAIRS_DIR).glob("*_pairs.jsonl")))

With **Save & Run All (Commit)** the outputs are attached to the saved version. Open notebook 03 next and attach
this notebook with **Add Input -> Notebooks**.

**Restart:** run all cells again. A finished generation file is reused and an interrupted one continues from its
last completed batch.